<a href="https://colab.research.google.com/github/morozovsolncev/ontology_of_synthesis/blob/main/matrix_5_(1_3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import networkx as nx

# =============================================================================
# ЭКСПЕРИМЕНТ 1.3 (УТОЧНЁННЫЙ): СМЕСЬ ДВУХ ТИПОВ НЕЭРМИТОВЫХ МАТРИЦ
# =============================================================================
# A-матрицы: epsilon_A = 1.0  (ближе к эрмитовым, "светлые")
# B-матрицы: epsilon_B = 10.0 (сильно неэрмитовы, "тёмные")
# Доля B варьируется с мелким шагом.
# =============================================================================

print("=" * 80)
print("ЭКСПЕРИМЕНТ 1.3 (УТОЧНЁННЫЙ): СМЕСЬ ДВУХ ТИПОВ НЕЭРМИТОВЫХ МАТРИЦ")
print("=" * 80)
print()

dim = 3
sigma = 4.0
percentile_threshold = 90
N_total = 600

epsilon_A = 1.0   # "светлая" материя
epsilon_B = 10.0  # "тёмная" материя

# Доли A
fractions_A = [0.0, 0.001, 0.002, 0.003, 0.005, 0.007, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.25, 0.30, 0.32, 0.34, 0.36, 0.38, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95,  0.96, 0.97, 0.98,  0.99, 1.0]


print("ПАРАМЕТРЫ ЭКСПЕРИМЕНТА:")
print(f"  dim = {dim}")
print(f"  N_total = {N_total}")
print(f"  sigma = {sigma}")
print(f"  percentile_threshold = {percentile_threshold}%")
print(f"  epsilon_A = {epsilon_A} (тип A, 'светлые')")
print(f"  epsilon_B = {epsilon_B} (тип B, 'тёмные')")
print(f"  fractions_A = {fractions_A[:5]}... (всего {len(fractions_A)})")
print()

# -----------------------------------------------------------------------------
# ФУНКЦИИ ГЕНЕРАЦИИ
# -----------------------------------------------------------------------------
def generate_random_hermitian(dim):
    real_part = np.random.randn(dim, dim)
    real_part = (real_part + real_part.T) / 2
    imag_part = np.random.randn(dim, dim)
    imag_part = (imag_part - imag_part.T) / 2
    return real_part + 1j * imag_part

def generate_non_hermitian(dim, epsilon):
    H = generate_random_hermitian(dim)
    A = generate_random_hermitian(dim)
    return H + 1j * epsilon * A

def commutator_norm(H1, H2):
    if H1.shape != H2.shape:
        return 1e10
    comm = H1 @ H2 - H2 @ H1
    return np.linalg.norm(comm, ord='fro')

def resonance_probability(H1, H2, sigma):
    return np.exp(-commutator_norm(H1, H2)**2 / sigma**2)

# ------------------------------------------------------------------------------------
# ОСНОВНОЙ ЦИКЛ
# ------------------------------------------------------------------------------------
print("-" * 80)
print(f"{'frac_A':>10} | {'N_A':>6} | {'N_B':>6} | {'size_main':>10} | {'count_A':>8} | {'count_B':>8} | {'frac_A_in':>10} | {'frac_B_in':>10}")
print("-" * 80)

results = []

for frac_A in fractions_A:
    N_A = int(N_total * frac_A)
    N_B = N_total - N_A

    np.random.seed(42)

    matrices = []
    types = []

    for _ in range(N_A):
        matrices.append(generate_non_hermitian(dim, epsilon_A))
        types.append('A')

    for _ in range(N_B):
        matrices.append(generate_non_hermitian(dim, epsilon_B))
        types.append('B')

    N = len(matrices)

    # Вычисление вероятностей
    prob_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            p = resonance_probability(matrices[i], matrices[j], sigma)
            prob_matrix[i, j] = p
            prob_matrix[j, i] = p

    all_probs = prob_matrix[np.triu_indices(N, k=1)]
    threshold = np.percentile(all_probs, percentile_threshold)

    # Граф
    G = nx.Graph()
    G.add_nodes_from(range(N))
    for i in range(N):
        for j in range(i+1, N):
            if prob_matrix[i, j] > threshold:
                G.add_edge(i, j)

    components = list(nx.connected_components(G))
    main_cluster = max(components, key=len) if components else set()
    size_main = len(main_cluster)

    count_A_in = sum(1 for v in main_cluster if types[v] == 'A')
    count_B_in = sum(1 for v in main_cluster if types[v] == 'B')

    frac_A_in = count_A_in / N_A if N_A > 0 else 0
    frac_B_in = count_B_in / N_B if N_B > 0 else 0

    print(f"{frac_A:10.3f} | {N_A:6d} | {N_B:6d} | {size_main:10d} | {count_A_in:8d} | {count_B_in:8d} | {frac_A_in:10.3f} | {frac_B_in:10.3f}")

    results.append({
        'frac_A': frac_A,
        'N_A': N_A,
        'N_B': N_B,
        'size_main': size_main,
        'count_A_in': count_A_in,
        'count_B_in': count_B_in,
        'frac_A_in': frac_A_in,
        'frac_B_in': frac_B_in
    })

print("-" * 80)

# -----------------------------------------------------------------------------
# ВЫВОДЫ
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ВЫВОДЫ ПО УТОЧНЁННОМУ ЭКСПЕРИМЕНТУ КАТАЛИЗА")
print("=" * 80)

# Ищем, где frac_B_in становится > 0.5 (B активно входят в кластер)
catalysis_point = None
for r in results:
    if r['frac_B_in'] > 0.5 and r['frac_A'] > 0 and r['frac_A'] < 1:
        catalysis_point = r['frac_A']
        break

if catalysis_point is not None:
    print(f"\n✓ КАТАЛИЗ ОБНАРУЖЕН: при доле A = {catalysis_point:.2f}")
    print(f"  доля B в главном кластере превысила 0.5")
else:
    print("\n○ ЯВНОГО КАТАЛИЗА (frac_B_in > 0.5) НЕ ОБНАРУЖЕНО.")
    # Найдём максимум frac_B_in
    max_frac_B = max(r['frac_B_in'] for r in results)
    print(f"  Максимальная доля B в главном кластере: {max_frac_B:.3f}")

# Также посмотрим, при какой доле A кластер максимален
max_size = max(r['size_main'] for r in results)
max_size_frac = [r['frac_A'] for r in results if r['size_main'] == max_size][0]
print(f"\n○ Максимальный размер главного кластера ({max_size}) достигается при доле A = {max_size_frac:.2f}")

print("\n" + "=" * 80)
print("ЭКСПЕРИМЕНТ ЗАВЕРШЁН")
print("=" * 80)

ЭКСПЕРИМЕНТ 1.3 (УТОЧНЁННЫЙ): СМЕСЬ ДВУХ ТИПОВ НЕЭРМИТОВЫХ МАТРИЦ

ПАРАМЕТРЫ ЭКСПЕРИМЕНТА:
  dim = 3
  N_total = 600
  sigma = 4.0
  percentile_threshold = 90%
  epsilon_A = 1.0 (тип A, 'светлые')
  epsilon_B = 10.0 (тип B, 'тёмные')
  fractions_A = [0.0, 0.001, 0.002, 0.003, 0.005]... (всего 35)

--------------------------------------------------------------------------------
    frac_A |    N_A |    N_B |  size_main |  count_A |  count_B |  frac_A_in |  frac_B_in
--------------------------------------------------------------------------------
     0.000 |      0 |    600 |         48 |        0 |       48 |      0.000 |      0.080
     0.001 |      0 |    600 |         48 |        0 |       48 |      0.000 |      0.080
     0.002 |      1 |    599 |        577 |        1 |      576 |      1.000 |      0.962
     0.003 |      1 |    599 |        577 |        1 |      576 |      1.000 |      0.962
     0.005 |      3 |    597 |        585 |        3 |      582 |      1.000 |      0.975